# Week 57, Day 2: Review: algorithms

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w057d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from collections import deque

graph = {"a": ["b", "c"], "b": ["d"], "c": ["d"], "d": []}
order, seen, queue = [], {"a"}, deque(["a"])
while queue:
    u = queue.popleft()
    order.append(u)
    for v in graph[u]:
        if v not in seen:
            seen.add(v)
            queue.append(v)
need = {u: 0 for u in graph}
for u in graph:
    for v in graph[u]:
        need[v] += 1
print(order, need)

In [ ]:
import heapq

def most_reliable(n, links, s, t):
    """links: (a, b, p) two-way, p the chance the link works. The best overall chance."""
    adj = [[] for _ in range(n)]
    for a, b, p in links:
        adj[a].append((b, p))
        adj[b].append((a, p))
    best, heap = [0.0] * n, [(-1.0, s)]
    best[s] = 1.0
    while heap:
        neg, u = heapq.heappop(heap)
        if -neg < best[u]:
            continue
        if u == t:
            return round(-neg, 6)
        for v, p in adj[u]:
            if -neg * p > best[v]:
                best[v] = -neg * p
                heapq.heappush(heap, (-best[v], v))
    return 0.0

print(most_reliable(3, [(0, 1, 0.5), (1, 2, 0.5), (0, 2, 0.2)], 0, 2))

In [ ]:
from functools import cmp_to_key

def largest_number(nums):
    words = [str(x) for x in nums]
    words.sort(key=cmp_to_key(lambda a, b: -1 if a + b > b + a else (1 if a + b < b + a else 0)))
    joined = "".join(words)
    return "0" if joined.startswith("0") else joined

print(largest_number([3, 30, 34, 5, 9]), largest_number([0, 0]))

In [ ]:
def total_hamming(nums):
    total = 0
    for b in range(max(nums, default=0).bit_length()):
        c = sum(x >> b & 1 for x in nums)
        total += c * (len(nums) - c)
    return total

print(total_hamming([4, 14, 2]), [bin(x) for x in (4, 14, 2)])

> **Key idea.** Graphs: BFS for fewest edges (mark when added), DFS and union-find for groups, Kahn's algorithm for orders and DAG DP, Dijkstra for non-negative costs and its variants (widest, most reliable), Bellman-Ford for negative edges or limited edges, Kruskal or Prim to connect cheaply. Greedy needs the right sort key and an exchange argument; backtracking needs early pruning. DP needs a state, a recurrence, an order and a reconstruction. Bit counts per position turn pair counts into O(n × bits). Write two approaches before choosing, and pick the simplest that's fast enough.

### Exercise: Safe starting points

In a directed graph given as a dict from node to a list of neighbours (every node is a key), a node is **safe** if every path from it eventually stops (reaches a node with no outgoing edges), so it can't wander into a cycle forever. Write `safe_nodes(graph)`: the sorted list of safe nodes. Reverse the edges and run Kahn's algorithm from the nodes with no outgoing edges.

In [ ]:
from collections import deque

def safe_nodes(graph):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_safe():\n    \"reverse topological order\"\n    g = {0: [1, 2], 1: [2, 3], 2: [5], 3: [0], 4: [5], 5: [], 6: []}\n    assert safe_nodes(g) == [2, 4, 5, 6] and safe_nodes({}) == [] and safe_nodes({0: [0]}) == []\n    rng = random.Random(1)\n    for _ in range(100):\n        n = rng.randint(1, 8)\n        g = {u: list({rng.randrange(n) for _ in range(rng.randint(0, 2))}) for u in range(n)}\n        def can_loop(u):\n            # u is unsafe if some path from it reaches a cycle\n            seen, stack = set(), [u]\n            reach = set()\n            while stack:\n                x = stack.pop()\n                for y in g[x]:\n                    if y not in reach:\n                        reach.add(y)\n                        stack.append(y)\n            nodes = reach | {u}\n            for x in reach:\n                inner, st = set(), [x]\n                while st:\n                    a = st.pop()\n                    for b in g[a]:\n                        if b == x:\n                            return True\n                        if b not in inner:\n                            inner.add(b)\n                            st.append(b)\n            return False\n        assert safe_nodes(g) == sorted(u for u in g if not can_loop(u))")

### Exercise: The most reliable route

Links between nodes 0 to n − 1 are `(a, b, p)`, two-way, where p (0 to 1) is the chance the link works. Write `most_reliable(n, links, s, t)`: the highest overall chance of getting from s to t (the product along a route), rounded to 6 places, `0.0` if unreachable and `1.0` if s is t, with Dijkstra on products.

In [ ]:
import heapq

def most_reliable(n, links, s, t):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_reliable():\n    \"Dijkstra on products\"\n    assert most_reliable(3, [(0, 1, 0.5), (1, 2, 0.5), (0, 2, 0.2)], 0, 2) == 0.25\n    assert most_reliable(3, [(0, 1, 0.5)], 0, 2) == 0.0 and most_reliable(2, [], 1, 1) == 1.0\n    rng = random.Random(2)\n    for _ in range(80):\n        n = rng.randint(2, 6)\n        links = [(a, b, round(rng.random(), 2)) for a in range(n) for b in range(a + 1, n) if rng.random() < 0.5]\n        s, t = rng.randrange(n), rng.randrange(n)\n        best = {}\n        for a, b, p in links:\n            best[(a, b)] = best[(b, a)] = max(p, best.get((a, b), 0))\n        want = 1.0 if s == t else 0.0\n        others = [v for v in range(n) if v not in (s, t)]\n        if s != t:\n            for r in range(len(others) + 1):\n                for mid in itertools.permutations(others, r):\n                    path = (s,) + mid + (t,)\n                    prob = 1.0\n                    for x, y in zip(path, path[1:]):\n                        prob *= best.get((x, y), 0)\n                    want = max(want, prob)\n        assert abs(most_reliable(n, links, s, t) - round(want, 6)) < 1e-6")

### Exercise: The largest number

Write `largest_number(nums)`: arrange non-negative whole numbers so that, written next to each other, they make the largest number, returned as a string (`"0"` rather than `"00"`). Sort with `functools.cmp_to_key`, putting a before b when `a + b > b + a` as strings.

In [ ]:
from functools import cmp_to_key

def largest_number(nums):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_largest():\n    \"a greedy comparison\"\n    assert largest_number([10, 2]) == \"210\" and largest_number([3, 30, 34, 5, 9]) == \"9534330\"\n    assert largest_number([0, 0]) == \"0\" and largest_number([1]) == \"1\" and largest_number([121, 12]) == \"12121\"\n    rng = random.Random(3)\n    for _ in range(150):\n        nums = [rng.choice([0, 1, 9, 10, 11, 12, 21, 3, 30, 34, 300, 5]) for _ in range(rng.randint(1, 6))]\n        want = max(int(\"\".join(map(str, p))) for p in itertools.permutations(nums))\n        assert largest_number(nums) == str(want)")

### Exercise: Restoring addresses

A string of digits had the dots removed from an internet address (four parts, each 0 to 255, no leading zeros except `0` itself). Write `addresses(digits)`: every possible address, sorted, by backtracking that prunes when the digits left can't fill the parts left (at least 1 and at most 3 digits per part).

In [ ]:
def addresses(digits):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_addresses():\n    \"backtracking with pruning\"\n    assert addresses(\"25525511135\") == [\"255.255.11.135\", \"255.255.111.35\"]\n    assert addresses(\"0000\") == [\"0.0.0.0\"] and addresses(\"101023\") == [\"1.0.10.23\", \"1.0.102.3\", \"10.1.0.23\", \"10.10.2.3\", \"101.0.2.3\"]\n    assert addresses(\"1\") == [] and addresses(\"1\" * 13) == []\n    rng = random.Random(4)\n    for _ in range(100):\n        s = \"\".join(rng.choice(\"0125\") for _ in range(rng.randint(4, 12)))\n        want = []\n        for a, b, c in itertools.combinations(range(1, len(s)), 3):\n            parts = [s[:a], s[a:b], s[b:c], s[c:]]\n            if all(len(p) <= 3 and (p == \"0\" or not p.startswith(\"0\")) and int(p) <= 255 for p in parts):\n                want.append(\".\".join(parts))\n        assert addresses(s) == sorted(want)")

### Exercise: Painting the houses

`cost[i][c]` is the cost of painting house i in colour c (at least two colours). Write `cheapest_paint(cost)`: the least total cost with no two neighbouring houses the same colour, in O(n × k) by keeping the previous row's smallest and second smallest. Then `paint_plan(cost)`: one cheapest list of colours.

In [ ]:
def cheapest_paint(cost):
    ___

def paint_plan(cost):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_paint():\n    \"a state machine per colour\"\n    c = [[17, 2, 17], [16, 16, 5], [14, 3, 19]]\n    assert cheapest_paint(c) == 10 and paint_plan(c) == [1, 2, 1] and cheapest_paint([]) == 0\n    rng = random.Random(5)\n    for _ in range(100):\n        n, k = rng.randint(1, 5), rng.randint(2, 4)\n        cost = [[rng.randint(1, 20) for _ in range(k)] for _ in range(n)]\n        best = min(sum(cost[i][p[i]] for i in range(n)) for p in itertools.product(range(k), repeat=n)\n                   if all(p[i] != p[i + 1] for i in range(n - 1)))\n        plan = paint_plan(cost)\n        assert cheapest_paint(cost) == best == sum(cost[i][plan[i]] for i in range(n))\n        assert all(plan[i] != plan[i + 1] for i in range(n - 1))\n    big = [[(i * 7 + c * 13) % 50 for c in range(30)] for i in range(5000)]\n    plan = paint_plan(big)\n    assert cheapest_paint(big) == sum(big[i][plan[i]] for i in range(5000))")

### Exercise: All the differences

Write `bits_up_to(n)`: a list of the number of 1 bits in each number from 0 to n, with `bits[i] = bits[i >> 1] + (i & 1)` (no `bin`), and `total_hamming(nums)`: the sum of the Hamming distances over every **pair** of numbers, in O(n × bits).

In [ ]:
def bits_up_to(n):
    ___

def total_hamming(nums):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_hamming():\n    \"bit counts\"\n    assert bits_up_to(5) == [0, 1, 1, 2, 1, 2] and bits_up_to(0) == [0]\n    assert bits_up_to(1000) == [bin(i).count(\"1\") for i in range(1001)]\n    assert total_hamming([4, 14, 2]) == 6 and total_hamming([]) == 0 and total_hamming([7]) == 0\n    rng = random.Random(6)\n    for _ in range(100):\n        nums = [rng.randint(0, 100) for _ in range(rng.randint(0, 10))]\n        assert total_hamming(nums) == sum(bin(a ^ b).count(\"1\") for i, a in enumerate(nums) for b in nums[i + 1:])\n    big = [rng.randint(0, 10 ** 9) for _ in range(50_000)]\n    assert total_hamming(big) > 0\n    assert \"bin(\" not in source.split(\"def bits_up_to\")[1].split(\"def total_hamming\")[0]")

### Bug hunt: Visited too late

In BFS, mark a node as seen when it's **added** to the queue. This marks it only when it's taken off, so a node reachable from several others is added several times, and the visit count prints `5` for 4 nodes. Mark on adding so it prints `4`.

In [ ]:
from collections import deque

graph = {"a": ["b", "c"], "b": ["d"], "c": ["d"], "d": []}
seen, queue, visits = set(), deque(["a"]), 0
while queue:
    u = queue.popleft()
    seen.add(u)
    visits += 1
    for v in graph[u]:
        if v not in seen:
            queue.append(v)
print(visits)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"mark when added\"\n    expect_output(\"4\")")

### Bug hunt: Neighbours in the same colour

Each house may not repeat its neighbour's colour, so f(i, c) must take the minimum over the previous row's **other** colours. This takes the minimum over all of them and prints `2`, painting both houses colour 0. Exclude colour c so it prints `6`.

In [ ]:
cost = [[1, 5, 9], [1, 6, 9]]
prev = cost[0]
for row in cost[1:]:
    prev = [row[c] + min(prev) for c in range(3)]
print(min(prev))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"different colours\"\n    expect_output(\"6\")")

### Bug hunt: The comparison back to front

For the largest number, a must come before b when `a + b` is **larger**. This returns −1 when it's smaller, which builds the smallest number, `"3033459"`. Flip the comparison so it prints `9534330`.

In [ ]:
from functools import cmp_to_key

words = ["3", "30", "34", "5", "9"]
words.sort(key=cmp_to_key(lambda a, b: -1 if a + b < b + a else 1))
print("".join(words))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"largest first\"\n    expect_output(\"9534330\")")

### Puzzle: Dijkstra with stale entries

Arrange the lines of Dijkstra's main loop (with `dist` and `heap` set up, `adj` a list of `(v, w)` lists). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the algorithm for cheapest routes with some negative edges: "Dijkstra", "Bellman-Ford" or "BFS"
q = ___   # bits_up_to(7)[-1]
r = ___   # largest_number([1, 20, 3])
s = ___   # how many valid addresses "1111" has
t = ___   # the total Hamming distance over pairs in [1, 2, 3]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"Bellman-Ford\" and q == 3 and r == \"3201\", \"p, q and r\"\n    assert s == 1 and t == 4, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: rising water, three ways

A square grid holds heights. Rain raises the water level by 1 every hour; at level t you can swim between neighbouring cells (up, down, left, right) if **both** are at most t. Starting top-left, what's the earliest t at which you can reach the bottom-right? Solve it three ways, which must agree:

- `earliest_binary(grid)`: binary search on t with a BFS check (Week 18 and Week 50).
- `earliest_dijkstra(grid)`: Dijkstra where a path's cost is its **largest** height (combine with `max` instead of `+`).
- `earliest_union(grid)`: sort cells by height and switch them on one at a time, joining each to its switched-on neighbours with union-find, until the two corners are joined.

In [ ]:
import heapq
from collections import deque

def earliest_binary(grid):
    ___

def earliest_dijkstra(grid):
    ___

def earliest_union(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef test_water():\n    \"three methods\"\n    g = [[0, 2], [1, 3]]\n    assert earliest_binary(g) == earliest_dijkstra(g) == earliest_union(g) == 3\n    g = [[0, 1, 2, 3, 4], [24, 23, 22, 21, 5], [12, 13, 14, 15, 16], [11, 17, 18, 19, 20], [10, 9, 8, 7, 6]]\n    assert earliest_binary(g) == earliest_dijkstra(g) == earliest_union(g) == 16\n    assert earliest_union([[5]]) == 5\n    rng = random.Random(7)\n    for _ in range(60):\n        n = rng.randint(1, 6)\n        cells = list(range(n * n))\n        rng.shuffle(cells)\n        grid = [cells[i * n:(i + 1) * n] for i in range(n)]\n        a = earliest_binary(grid)\n        assert a == earliest_dijkstra(grid) == earliest_union(grid)\n    n = 120\n    cells = list(range(n * n))\n    rng.shuffle(cells)\n    big = [cells[i * n:(i + 1) * n] for i in range(n)]\n    t0 = time.perf_counter()\n    assert earliest_binary(big) == earliest_dijkstra(big) == earliest_union(big)\n    assert time.perf_counter() - t0 < 10")

## How to review

The same habits as yesterday. Today's extra one: for each problem, write **two** possible approaches before choosing, with their costs. The exam will reward you for picking the simpler one that's fast enough.

## Graphs

- **BFS** (Week 50): fewest edges, level by level, with a queue; mark nodes when they're **added**, not when they're taken off.
- **DFS** (Week 50): reachability, components, cycles; with a stack to avoid recursion limits.
- **Topological order** (Week 50): Kahn's algorithm on in-degrees; fewer nodes out than in means a cycle. DAG DP runs in this order (Week 54).
- **Dijkstra** (Week 51): cheapest paths with non-negative weights; a heap of `(cost, node)`, skipping stale entries. Variants change the combine rule: widest path (max of min), most reliable path (max of products).
- **Bellman-Ford** (Week 51): negative weights, negative cycles, at most k edges.
- **Union-find and minimum spanning trees** (Week 51): connect cheaply, merge groups.

Dijkstra works for products of chances because multiplying by a number at most 1 never increases the value, just as adding a non-negative cost never decreases a distance.

## Greedy and backtracking

**Greedy** (Week 52): sort by the right key and take what fits, justified by an exchange argument. For "arrange numbers to make the largest number", the key is a **comparison**: put a before b when the digits ab form a larger number than ba. The exchange argument: in any arrangement, swapping a neighbouring pair that's in the wrong order makes the number larger, so the best arrangement has no such pair.

**Backtracking** (Week 52): choose, explore, unchoose, pruning as early as possible. Splitting a string of digits into four parts that make a valid internet address (each part 0 to 255, no leading zeros) is a small search with strong pruning: each part has at most 3 digits, and what's left must fit in the parts left.

## Dynamic programming

From Weeks 53 and 54: spot overlapping choices; design the state as exactly what the rest of the answer depends on; write the recurrence; fill in an order where ingredients come first (or memoise); reconstruct with a parent table. The patterns: best ending here, prefix splits, small state machines, grids, knapsacks, two strings, intervals, trees, DAGs, bitmasks, digits.

Painting houses in a row, each in one of k colours with given costs, no two neighbours alike: a state machine with one value per colour. f(i, c) = cost[i][c] + the smallest f(i − 1, c′) over c′ ≠ c. Keeping the smallest and second smallest of the previous row makes each row O(k) instead of O(k²).

## Bits and maths

From Week 55: `x & (x − 1)`, `x & −x`, XOR cancelling; counting bits for every number up to n with `bits[i] = bits[i >> 1] + (i & 1)`; modular arithmetic, inverses and binomials; inclusion-exclusion; seeded randomness. A typical counting trick: the total Hamming distance over all **pairs** of numbers, bit by bit: if c of the n numbers have bit b set, that bit contributes c × (n − c) differing pairs.

## Practice

## Quiz

## Daily challenge

## Before you go

For each family on today's list, write the name of one problem you could solve with it, from memory. Tomorrow: the practice exam, in the same format as Friday's.